In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Loading the dataset

### Transformations => Tensors => Dataset => DataLoader

In [2]:
# load image -> apply transformations -> dataset of all images
class ImageProcessor:

    def __init__(self, root_dir_path, transformations=None):
        self.root_dir_path  = root_dir_path
        self.transformations = transformations

        self.all_img_paths = [os.path.join(root_dir_path, img_path) for img_path in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self, key):
        img_path = self.all_img_paths[key]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img

In [3]:
root_dir_path = "dataset/img_align_celeba"
transformations = transforms.Compose([
    transforms.CenterCrop(178), # 178 x 218 => 178 x 178
    transforms.Resize(64), # 178 x 178 => 64 x 64
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Each pixel normalized => [-1, 1]
])

In [4]:
dataset = ImageProcessor(root_dir_path, transformations)
print(f"loaded {len(dataset)} images")

loaded 202599 images


In [5]:
data_loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

# Defining & Training the GAN

### Generator Network

In [6]:
class Generator(nn.Module):

    def __init__(self, z_dim=100, img_channels=3):
        super().__init__()
        self.img_channels = img_channels

        # Fully connected layer
        self.model = nn.Sequential(
            nn.Linear(z_dim, 256), # 100 => 256
            nn.ReLU(),

            nn.Linear(256, 512), # 256 => 512
            nn.ReLU(),
            
            nn.Linear(512, 1024), # 512 => 1024
            nn.ReLU(),

            nn.Linear(1024, 64 * 64 * img_channels), # 1024 => 64 * 64 * 3 (depends on argument)
            nn.Tanh() # [-1, 1] to match structure of original images
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), self.img_channels, 64, 64)
        return img


### Discriminator Network

In [7]:
class Discriminator(nn.Module):

    def __init__(self, img_channels=3):
        super().__init__()

        # Fully connected layer
        self.model = nn.Sequential(
            nn.Flatten(), # 4D (batch_size * img_channel * height * width) => 1D 

            nn.Linear(img_channels * 64 * 64, 1024), # 3 * 64 * 64 => 1024
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(1024, 512), # 1024 => 512
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Linear(512, 256), # 512 => 256
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(256, 1), # 246 => 1
            nn.Sigmoid()
        )

    def forward(self, img):
        return self.model(img)


### Training

In [8]:
if torch.backends.mps.is_available():
    device="mps"
elif torch.cuda.is_available():
    device="cuda"
else:
    device="cpu"
device = torch.device(device)

print(f"Device: {device}")

Device: cpu


In [9]:
GAN_loss = nn.BCELoss()

generator = Generator().to(device)
discriminator = Discriminator().to(device)

g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [10]:
class Trainer():

    def __init__(self, generator: Generator, discriminator: Discriminator, g_optimizer: optim,  d_optimizer: optim, GAN_loss: nn.BCELoss):
        self.generator = generator
        self.discriminator = discriminator
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer
        self.GAN_loss = GAN_loss

    def train(self, data_loader, epochs = 10):

        for epoch in range(epochs):
            for i, imgs in enumerate(data_loader):
                real_imgs = imgs.to(device)
                batch_size = real_imgs.size(0)

                real_labels = torch.ones(batch_size, 1).to(device)
                fake_labels = torch.zeros(batch_size, 1).to(device)

                # Train the discriminator
                self.d_optimizer.zero_grad()

                fake_imgs = self.generator(torch.randn(batch_size, 100).to(device))

                real_loss = self.GAN_loss(self.discriminator(real_imgs), real_labels)
                fake_loss = self.GAN_loss(self.discriminator(fake_imgs.detach()), fake_labels)
                d_loss = (real_loss + fake_loss) / 2

                d_loss.backward()
                self.d_optimizer.step()

                # Train the generator
                self.g_optimizer.zero_grad()

                g_loss = self.GAN_loss(self.discriminator(fake_imgs), real_labels) # calculate loss
                g_loss.backward() # backward propagation
                self.g_optimizer.step() # optimization

                if i % 50 == 0:
                    print(f"Epoch: {epoch + 1} Batch: {i + 1} G_loss: {g_loss} D_loss: {d_loss}")

            torch.save(self.generator.state_dict(), f"/data/generator_{epoch}.pt")

            self.save_generated_image(epoch, device)

    def save_generated_image(self, epoch: int, device: torch.device, num_images: int = 8):
        z = torch.randn(num_images, 100).to(device)
        fake_images = self.generator(z).detach().cpu()

        grid = torchvision.utils.make_grid(fake_images, nrow=4, normalize=True)

        plt.imshow(np.transpose(grid, (1, 2, 0))) # plt expects height x width x channels
        plt.title(f"Epoch: {epoch}")
        plt.show()

        


In [11]:
trainer = Trainer(generator, discriminator, g_optimizer, d_optimizer, GAN_loss)

In [12]:
# trainer.train(data_loader)
# Run on Google Collab

# Testing

In [14]:
generator.load_state_dict(torch.load("saved_models/generator_9.pt", map_location=device))

<All keys matched successfully>

In [20]:
img = generator(torch.randn(1, 100).to(device))[0]

In [21]:
torchvision.utils.save_image(img, "generated_image.png")